## Exercise 1

As a good computer science engineer, you know that your first step should be to load and understand the data you are about to work on. 

- Load the file containing the mails (use pandas)
- Show the following statistics about this file:
    - total number of mails
    - total number of mails corresponding to spams
    - total number of mails not corresponding to spams (hams)
    - the mean length of the mails

In [1]:
# dataset: https://www.kaggle.com/datasets/venky73/spam-mails-dataset?resource=download

import pandas as pd

data = pd.read_csv(
    "TD6-files/spams.csv",
    sep=",",
    header="infer",
    index_col=0,
    encoding="utf-8"
)

data

,label,text,label_num
605,ham,Subject: enron methanol ; meter # : 988291\r\n...,0
2349,ham,"Subject: hpl nom for january 9 , 2001\r\n( see...",0
3624,ham,"Subject: neon retreat\r\nho ho ho , we ' re ar...",0
4685,spam,"Subject: photoshop , windows , office . cheap ...",1
2030,ham,Subject: re : indian springs\r\nthis deal is t...,0
...,...,...,...
1518,ham,Subject: put the 10 on the ft\r\nthe transport...,0
404,ham,Subject: 3 / 4 / 2000 and following noms\r\nhp...,0
2933,ham,Subject: calpine daily gas nomination\r\n>\r\n...,0
1409,ham,Subject: industrial worksheets for august 2000...,0


In [2]:
def count_mails(label_list):
    """Count the number of mails, hams and spams in dataframe"""
    hams = 0
    spams = 0
    
    for i,label in enumerate(label_list):
        if label == "ham" or label == 0:
            hams += 1
        elif label == "spam" or label == 1:
            spams += 1
        else:
            print(f"Bad label for mail of ID {i}: {label}")
    
    print(f"Number of mails:\t {len(label_list)}")
    print(f"Number of hams: \t {hams}")
    print(f"Number of spams:\t {spams}")

count_mails(data["label"])

mails_len = [len(i) for i in data["text"]]
mean_len = round(sum(mails_len)/len(mails_len),2)

print(f"Mean mail length:\t {mean_len}")

Number of mails:	 5171
Number of hams: 	 3672
Number of spams:	 1499
Mean mail length:	 1048.39


## Exercise 2

You wonder if your predecessor did the basics steps when creating the dataset you are working on. For instance, did he remove every duplicates in this dataset ?

- Search for mails with the exact same text (we call this kind of search an exact match search)
    - Ouch, you found around 178 duplicates ! Clearly, this dataset was not well-handled by your predecessor
- To investigate further, you chose to perform a cosine similarity test between every mails to find every duplicates:
    - import the required functions from scikit-learn
    - perform a similarity check between each possible pair of mail (remember how we did it during the lecture)
- Remove every mail with a cosine similarity of 0.9 or more with another mail
    - At least ! You removed around 1,421 mails from this dataset, it will be easier to process now
- Make sure the dataset is still usable - print the total number of mails, spams and hams

In [3]:
seen = []
duplicates = []

for i,mail in enumerate(data["text"]):
    if mail not in seen:
        seen.append(mail)
    else:
        duplicates.append(i)

print(len(duplicates))

178


In [4]:
from sklearn.metrics import pairwise_kernels
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(data["text"])

sim_matrice = pairwise_kernels(X,metric="cosine")

print(sim_matrice)

[[1.         0.05730683 0.27399551 ... 0.24070533 0.2262637  0.26647819]
 [0.05730683 1.         0.04336691 ... 0.02353104 0.11964501 0.02657157]
 [0.27399551 0.04336691 1.         ... 0.16500977 0.38447783 0.52511545]
 ...
 [0.24070533 0.02353104 0.16500977 ... 1.         0.12669172 0.13568071]
 [0.2262637  0.11964501 0.38447783 ... 0.12669172 1.         0.2320784 ]
 [0.26647819 0.02657157 0.52511545 ... 0.13568071 0.2320784  1.        ]]


In [5]:
# Careful: when comparing mails with a cosine similarity matrice, you will compare a mail with itself in each row !

ids_to_remove = set()

for i,sim_row in enumerate(sim_matrice): # if j == i, do not process
    for j,score in enumerate(sim_row):

        if score > 0.9 and i != j:
            ids_to_remove.add(j)

print(len(ids_to_remove))
data = data.drop(list(ids_to_remove))

1421


In [6]:
count_mails(data["label"])

Number of mails:	 3750
Number of hams: 	 2657
Number of spams:	 1093


## Exercise 3

Your data is now ready to be used to train a Machine Learning Model ! The goal for this model is to classify each mail in the "spam" or "ham" category. 

- Separate the text and the label of the mails in two lists
- Use scikit-learn's "train_test_split" function to create a train partition and a test partition
- Check for the number of mails, hams and spams in each partition
- A Machine Learning Model usually take vectors as inputs:
    - you MUST apply a vectorizer to your data
    - first step: import and initiate a vectorizer (see the last lecture's notebooks)
    - second step: apply it on each partition (on the texts of the mails) 
- Now is the time to train your first model: a linear regression should do the trick
    - import the model with scikit-learn
    - train it on the train partition and check the accuracy
    - test it on the test partition and check the accuracy
- You want to compare your model with more models
    - chose two models from the following list: https://scikit-learn.org/stable/supervised_learning.html
    - reproduce the experiment you made for the linear regression, but with these models instead

In [16]:
from sklearn.model_selection import train_test_split

x = list(data["text"])
y = list(data["label_num"])

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.3, random_state=42
)

print("Train partition:")
count_mails(y_train)

print("\nTest partition:")
count_mails(y_test)


# Step 3: initialize a Machine Learning model


# ex3 fin: il faut faire de la cross validation sur chaque modele
# ex4: il faut faire du matplotlib sur les resultats : 3 courbes, un par modele. Chaque point est une run sur la cross validation
# ex4 suite: afficher sur un graph le nb de spams et de hams
# ex4 suite: afficher la loi de zipf sur tous les mials

Train partition:
Number of mails:	 2625
Number of hams: 	 1869
Number of spams:	 756

Test partition:
Number of mails:	 1125
Number of hams: 	 788
Number of spams:	 337


In [17]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()

x_train = vectorizer.fit_transform(x_train)
x_test = vectorizer.transform(x_test)

In [18]:
# Step 3: initialize a Machine Learning model

from sklearn.linear_model import LinearRegression

model = LinearRegression()    # we initialize the model
model.fit(x_train, y_train)   # we give training data to the model
model.score(x_train, y_train) # we check the accuracy of our model on the train partition 

0.9999276553803389

In [19]:
print(model.predict(x_test))
print(model.score(x_test,y_test))

[ 0.06316865  0.85623712 -0.39497349 ... -1.98152967 -0.72115261
  1.1461072 ]
-6.643919768117874
